# Player detection + Jersey number recognition (Kaggle)

**Settings trước khi chạy:**
- *Accelerator*: **GPU T4 x2** (hoặc T4). Tránh P100: torch bản mới trên Kaggle không còn hỗ trợ.
- *Internet*: **On** (để clone repo, pip install, tải weights).
- *Add Input*: thêm dataset chứa video bóng đá (`.mp4`), ví dụ upload clip của bạn thành 1 Kaggle Dataset.

Pipeline (mặc định): yolo26s-soccer (detect player/goalkeeper) → ByteTrack → legibility classifier (ResNet34) → crop thân áo theo bbox → PARSeq đọc số → bỏ phiếu theo track.

In [ ]:
# 1. Clone repo  
REPO_URL = "https://github.com/nvdat16/SoccerPipeline.git"
!git clone -q $REPO_URL /kaggle/working/SoccerPipeline
%cd /kaggle/working/SoccerPipeline

In [ ]:
# 2. Cài dependencies + tải weights (torch/numpy/opencv dùng bản sẵn của Kaggle)
!bash scripts/setup.sh
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. (Tuỳ chọn) weights từ Kaggle Dataset thay vì Google Drive
#    Nếu cell 2 báo Google Drive hết lượt tải: upload 2 file .pth/.ckpt lên 1 Kaggle Dataset rồi bỏ comment:
# import os; os.environ["MODELS_DIR"] = "/kaggle/input/jersey-weights"

In [ ]:
# 4. Chọn video input (mặc định: video đầu tiên tìm thấy trong /kaggle/input)
import glob
videos = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True) +
                glob.glob("/kaggle/input/**/*.mkv", recursive=True) +
                glob.glob("/kaggle/input/**/*.avi", recursive=True))
print("\n".join(videos) or "Chưa có video — Add Input một dataset chứa video")
VIDEO = videos[0] if videos else None
print("=> VIDEO =", VIDEO)

In [ ]:
# 5. Chạy pipeline — chọn 1 luồng:
#   run_bbox.sh          [khuyến nghị] yolo26s-soccer + crop thân áo theo bbox (không pose)
#   run_pose.sh          yolo26s-soccer + yolo26n-pose cắt theo vai/hông (giống paper)
#   run_coco.sh          yolo11m-pose COCO: bắt cả cảnh cận nhưng lẫn khán giả
#   run_no_legibility.sh debug: bỏ legibility (PARSeq sẽ bịa số)
#   compare.sh           chạy tất cả luồng + in bảng so sánh
#   diagnose.sh          chẩn đoán vì sao số áo không được nhận
# MAX_FRAMES=750: test nhanh 15s đầu (50fps). Bỏ đi để chạy cả video.
CASE = "bbox"
!MAX_FRAMES=750 bash scripts/run_$CASE.sh "$VIDEO" /kaggle/working/outputs/$CASE

In [ ]:
# 6. Kết quả theo track
import json, pandas as pd
from pathlib import Path
out_dir = Path("/kaggle/working/outputs") / CASE / Path(VIDEO).stem
res = json.load(open(out_dir / "tracks.json"))
df = pd.DataFrame.from_dict(res["tracks"], orient="index")
df.index.name = "track_id"
df.sort_values(["jersey_number", "weight"], ascending=[False, False]).head(40)

In [ ]:
# 7. Xem video kết quả
from IPython.display import Video
Video(str(out_dir / f"{Path(VIDEO).stem}_jersey.mp4"), embed=True, width=960)

In [ ]:
# 8. Tốc độ suy luận từng bước (cũng được in ở cuối output cell 5)
timing = json.load(open(out_dir / "timing.json"))
print(f"device={timing['device']}  frames={timing['frames']}  analyse={timing['analyse_fps']} fps  "
      f"end-to-end={timing['fps']} fps")
pd.DataFrame.from_dict(timing["steps"], orient="index")[
    ["calls", "items", "total_s", "ms_per_call_p50", "ms_per_item", "ms_per_frame", "pct_wall"]]

**Output** ở `/kaggle/working/outputs/<tên video>/`:
- `<tên>_jersey.mp4`: video có bbox + số áo
- `tracks.json`: số áo cuối cùng của từng track + phiếu bầu
- `detections.csv`: bbox từng frame (frame, track_id, x1..y2, jersey_number)- `timing.json`: thời gian từng bước (ms/frame, ms/crop, % tổng thời gian)
